# 04 Sequential & Replenishment MBA

**Client deliverable for LushProtein — adding the dimension classic MBA ignores: time.**

Notebooks `01`–`03` analyse what customers buy **in the same basket**. But supplements are *consumed on a clock* (a tub lasts ~6–8 weeks) and a customer's relationship unfolds **across orders**. Two questions that within-basket MBA cannot answer, but that drive retention:

1. **Sequence — "what do they buy *next*?"** After a customer's first purchase, which product/category do they graduate to on their *second* order? This is the moment the EDA shows is decisive: most repeat customers place order #2 within ~50 days, and reaching 2–3 categories is what lifts repeat rate from 24% to 65%.
2. **Timing — "*when* are they due?"** Every hero SKU has a measured reorder interval (`12_reorder_interval_by_sku.csv`). Pairing affinity with the consumption clock turns a static "people who buy X also buy Y" rule into a *triggered* "email Y three days before X runs out" flow — the mechanic DTC brands run in Klaviyo.

This notebook produces three artefacts: **sequential (next-purchase) rules**, a **gateway-product scorecard**, and a **replenishment-trigger calendar**.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "EDA" / "outputs").exists() and (candidate / "product_mba").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing EDA/outputs and product_mba")

PROJECT_ROOT = find_project_root()
EDA_OUTPUTS = PROJECT_ROOT / "EDA" / "outputs"
MBA_OUTPUTS = PROJECT_ROOT / "product_mba" / "outputs"
MBA_OUTPUTS.mkdir(exist_ok=True)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
print("Project root found")

Project root found


## 1. Sequential rules — first order → second order

We order each customer's orders by date and ask: *given category/handle **A** in the first order, what is the chance category **B** appears in the second order, versus B's base rate across all second orders?* `lift > 1` means buying A makes B a more likely next step than chance — a genuine **graduation path**, not just a co-purchase.

In [2]:
lines = pd.read_parquet(EDA_OUTPUTS / "lines.parquet")
lines["order_date"] = pd.to_datetime(lines["order_date"], utc=True)
lines["customer_id"] = lines["customer_id"].astype(str)

import re
# Some Line: Product Handle values are campaign/promo tags, not real products
# (e.g. "sg-nov-2025-victory-tag", gift-with-purchase markers). Drop them.
_JUNK = re.compile(r"(tag|victory|promo|gift|20[0-9][0-9]|free|bonus|sample-gift)", re.I)

def clean(v):
    if pd.isna(v):
        return pd.NA
    t = str(v).strip()
    if not t or t.lower() in {"nan", "none", "null", "unknown"}:
        return pd.NA
    if _JUNK.search(t):
        return pd.NA
    return t

def sequential_rules(level_col, label, min_co=15, min_first=40):
    df = lines[["customer_id", "order_id", "order_date", level_col]].copy()
    df[level_col] = df[level_col].map(clean)
    df = df.dropna(subset=[level_col])
    # rank orders within each customer by date
    order_rank = (df[["customer_id", "order_id", "order_date"]].drop_duplicates()
                    .sort_values(["customer_id", "order_date"]))
    order_rank["seq"] = order_rank.groupby("customer_id").cumcount() + 1
    df = df.merge(order_rank[["customer_id", "order_id", "seq"]], on=["customer_id", "order_id"])

    first = (df[df["seq"] == 1].groupby("customer_id")[level_col]
               .apply(lambda s: sorted(set(s))).rename("first_items"))
    second = (df[df["seq"] == 2].groupby("customer_id")[level_col]
                .apply(lambda s: sorted(set(s))).rename("second_items"))
    j = pd.concat([first, second], axis=1).dropna()
    n_repeat = len(j)
    if n_repeat == 0:
        return pd.DataFrame()

    # base rate of each B appearing in a second order
    base_b = (j["second_items"].explode().value_counts() / n_repeat).rename("consequent_base_rate")

    rows = []
    for _, r in j.iterrows():
        for a in r["first_items"]:
            for b in r["second_items"]:
                rows.append((a, b))
    trans = pd.DataFrame(rows, columns=["antecedent", "consequent"])
    cnt = trans.groupby(["antecedent", "consequent"]).size().rename("co_customers").reset_index()
    first_counts = j["first_items"].explode().value_counts().rename("antecedent_first_customers")
    cnt = cnt.merge(first_counts, left_on="antecedent", right_index=True)
    cnt = cnt.merge(base_b, left_on="consequent", right_index=True)
    cnt["next_order_confidence"] = cnt["co_customers"] / cnt["antecedent_first_customers"]
    cnt["next_order_lift"] = cnt["next_order_confidence"] / cnt["consequent_base_rate"]
    cnt["level"] = label
    cnt["repeat_customers_analyzed"] = n_repeat
    out = cnt[(cnt["co_customers"] >= min_co) &
              (cnt["antecedent_first_customers"] >= min_first)].copy()
    return out.sort_values(["next_order_lift", "co_customers"], ascending=[False, False])

seq_cat = sequential_rules("product_category", "category")
seq_handle = sequential_rules("Line: Product Handle", "handle")
sequential = pd.concat([seq_cat, seq_handle], ignore_index=True)
sequential.to_csv(MBA_OUTPUTS / "mba_sequential_rules.csv", index=False)

print(f"Sequential rules: {len(seq_cat)} category, {len(seq_handle)} handle")
print("\nTop category graduation paths (first order -> second order):")
display(seq_cat.head(10).reset_index(drop=True))
print("Top handle graduation paths:")
display(seq_handle.head(12).reset_index(drop=True))

Sequential rules: 33 category, 76 handle

Top category graduation paths (first order -> second order):


,antecedent,consequent,co_customers,antecedent_first_customers,consequent_base_rate,next_order_confidence,next_order_lift,level,repeat_customers_analyzed
0,Soy Protein,Soy Protein,91,143,0.063,0.636,10.088,category,2267
1,Collagen Glow,Collagen Glow,209,312,0.140,0.670,4.775,category,2267
2,Lean Protein,Lean Protein,371,541,0.259,0.686,2.648,category,2267
3,Clear Protein,Clear Protein,365,621,0.228,0.588,2.582,category,2267
4,Accessories,Accessories,92,434,0.125,0.212,1.698,category,2267
5,Clear Protein,Accessories,121,621,0.125,0.195,1.561,category,2267
6,Other,Other,960,1198,0.515,0.801,1.555,category,2267
7,Accessories,Lean Protein,157,434,0.259,0.362,1.397,category,2267
8,Accessories,Clear Protein,133,434,0.228,0.306,1.346,category,2267
9,Lean Protein,Accessories,86,541,0.125,0.159,1.273,category,2267


Top handle graduation paths:


,antecedent,consequent,co_customers,antecedent_first_customers,consequent_base_rate,next_order_confidence,next_order_lift,level,repeat_customers_analyzed
0,better-whey-protein-elite,better-whey-protein-elite,28,45,0.025,0.622,24.700,handle,2223
1,prime-whey-isolate,prime-whey-isolate,96,150,0.057,0.640,11.203,handle,2223
2,soy-protein-isolate,soy-protein-isolate,91,143,0.065,0.636,9.824,handle,2223
3,plant-protein,plant-protein,144,206,0.089,0.699,7.848,handle,2223
4,lushprotein-lean-protein-40g-single-serve,lushprotein-lean-protein-40g-single-serve,22,121,0.034,0.182,5.318,handle,2223
5,collagen-glow,collagen-glow,209,302,0.141,0.692,4.915,handle,2223
6,micronized-creatine-monohydrate,micronized-creatine-monohydrate,104,194,0.112,0.536,4.767,handle,2223
7,better-whey,better-whey,393,559,0.216,0.703,3.256,handle,2223
8,lushprotein-lean-protein-40g-single-serve,lean-protein,82,121,0.228,0.678,2.971,handle,2223
9,clear-protein,discovery-sampler,19,552,0.012,0.034,2.834,handle,2223


In [3]:
# How different is the sequential view from the within-basket view?
basket_cat = pd.read_csv(MBA_OUTPUTS / "mba_rules_category.csv")
basket_pairs = set(zip(basket_cat["antecedent"], basket_cat["consequent"]))
seq_pairs = set(zip(seq_cat["antecedent"], seq_cat["consequent"]))
print("Category pairs only visible in the SEQUENTIAL (next-purchase) view, "
      "not in within-basket rules:")
for a, b in sorted(seq_pairs - basket_pairs):
    print(f"   {a}  ->(next order)->  {b}")

Category pairs only visible in the SEQUENTIAL (next-purchase) view, not in within-basket rules:
   Accessories  ->(next order)->  Accessories
   Accessories  ->(next order)->  Collagen Glow
   Accessories  ->(next order)->  Other
   Accessories  ->(next order)->  Soy Protein
   Clear Protein  ->(next order)->  Clear Protein
   Clear Protein  ->(next order)->  Collagen Glow
   Clear Protein  ->(next order)->  Lean Protein
   Clear Protein  ->(next order)->  Other
   Clear Protein  ->(next order)->  Soy Protein
   Collagen Glow  ->(next order)->  Accessories
   Collagen Glow  ->(next order)->  Clear Protein
   Collagen Glow  ->(next order)->  Collagen Glow
   Collagen Glow  ->(next order)->  Lean Protein
   Collagen Glow  ->(next order)->  Other
   Collagen Glow  ->(next order)->  Soy Protein
   Lean Protein  ->(next order)->  Clear Protein
   Lean Protein  ->(next order)->  Collagen Glow
   Lean Protein  ->(next order)->  Lean Protein
   Lean Protein  ->(next order)->  Other
   Other  -

## 2. Gateway-product scorecard — which entry point breeds loyalty?

Not all first purchases are equal. The EDA measured, per acquisition product and per first flavor, the downstream repeat rate, subscription rate and LTV. We rank these as *gateway* products — the entry SKUs LushProtein should push hardest in acquisition, because they pre-load retention.

In [4]:
gateway_cat = pd.read_csv(EDA_OUTPUTS / "04_repeat_by_first_product.csv")
gateway_cat = gateway_cat.sort_values("repeat_rate", ascending=False)
print("Gateway by first CATEGORY (entry product -> downstream loyalty):")
display(gateway_cat[["first_product_cat", "customers", "repeat_rate",
                     "avg_ltv", "avg_total_orders", "median_days_2nd"]].reset_index(drop=True))

flavor = pd.read_csv(EDA_OUTPUTS / "12_first_flavor_loyalty_min30.csv")
flavor["gateway_score"] = flavor["repeat_rate"] * (1 + flavor["pct_subscribed"])
flavor = flavor.sort_values("gateway_score", ascending=False)
keep = ["first_handle", "first_variant", "customers", "repeat_rate",
        "pct_subscribed", "avg_ltv", "avg_orders", "gateway_score"]
flavor[keep].to_csv(MBA_OUTPUTS / "gateway_product_scorecard.csv", index=False)
print("\nTop gateway FLAVORS/SKUs (min 30 customers, ranked by repeat x subscription):")
display(flavor[keep].head(15).reset_index(drop=True))

# Does the Discovery Sampler / sachet entry behave like a gateway?
samp = flavor[flavor["first_handle"].astype(str).str.contains("sampler|discovery|single-serve|sachet",
                                                              case=False, na=False)]
if len(samp):
    print("\nSampler / single-serve entry points:")
    display(samp[keep].reset_index(drop=True))
else:
    print("\n(No distinct sampler/single-serve handle cleared the 30-customer threshold "
          "in 12_first_flavor_loyalty_min30.csv.)")

Gateway by first CATEGORY (entry product -> downstream loyalty):


,first_product_cat,customers,repeat_rate,avg_ltv,avg_total_orders,median_days_2nd
0,Unknown,6908,0.406,336.054,2.337,52.000
1,Collagen Glow,439,0.312,130.116,1.909,48.000
2,Other,2502,0.251,139.257,1.869,53.000
3,Lean Protein,1324,0.232,102.070,1.444,36.000
4,Clear Protein,1550,0.225,109.860,1.472,44.000
5,Accessories,696,0.221,69.113,1.395,20.000
6,Soy Protein,361,0.208,89.809,1.454,64.000



Top gateway FLAVORS/SKUs (min 30 customers, ranked by repeat x subscription):


,first_handle,first_variant,customers,repeat_rate,pct_subscribed,avg_ltv,avg_orders,gateway_score
0,better-whey,1KG Pack / Cocoa Dinosaur,30,1.000,0.300,"1,633.192",8.633,1.300
1,collagen-glow,300g Pack,31,1.000,0.161,883.699,7.065,1.161
2,collagen-glow,300g Pack (30 servings) / Natural (Unflavoured),36,0.444,0.222,177.593,2.361,0.543
3,collagen-glow,300g,38,0.421,0.211,124.157,2.211,0.510
4,lean-protein,1kg Pack (25 serves) / Taro,39,0.385,0.205,119.755,1.974,0.464
5,better-whey,1kg Pack (40 servings) / Natural (Unflavoured),43,0.372,0.116,158.085,1.721,0.415
6,lean-protein,1kg Pack (25 servings) / Thai Milk Tea,192,0.349,0.177,127.784,1.818,0.411
7,collagen-glow,300g Pack,76,0.342,0.197,110.263,2.329,0.410
8,lushprotein-lean-protein-40g-single-serve,40g Sachet / Taro,32,0.344,0.156,70.614,1.594,0.397
9,better-whey,1kg Pack (40 servings) / Cocoa Dinosaur,32,0.312,0.156,194.936,2.312,0.361



Sampler / single-serve entry points:


,first_handle,first_variant,customers,repeat_rate,pct_subscribed,avg_ltv,avg_orders,gateway_score
0,lushprotein-lean-protein-40g-single-serve,40g Sachet / Taro,32,0.344,0.156,70.614,1.594,0.397
1,lushprotein-lean-protein-40g-single-serve,40g Sachet / Thai Milk Tea,32,0.250,0.062,58.002,1.375,0.266
2,discovery-sampler,6 Sachets Variety,54,0.241,0.074,109.553,1.463,0.259
3,lushprotein-lean-protein-40g-single-serve,5 x 40g Sachet (5 servings) *Most Popular* / Taro,44,0.227,0.023,62.244,1.273,0.232
4,lushprotein-lean-protein-40g-single-serve,1 x 40g Sachet (1 serving) / Thai Milk Tea,36,0.222,0.028,78.698,1.361,0.228
5,lushprotein-lean-protein-40g-single-serve,1 x 40g Sachet (1 serving) / Taro,39,0.179,0.000,80.416,1.256,0.179
6,lushprotein-lean-protein-40g-single-serve,5 x 40g Sachet (5 servings) *Most Popular* / T...,39,0.128,0.077,45.729,1.128,0.138
7,clear-protein-25g-single-sachet,1 x 25g (1 serving) / Peach,40,0.125,0.050,54.423,1.175,0.131
8,discovery-sampler,5 Sachets Variety,53,0.057,0.038,52.896,1.057,0.059


## 3. Replenishment-trigger calendar — pair affinity with the consumption clock

For each hero handle we read its measured reorder interval, then attach the best **retention-scored** cross-sell consequent (from notebook `03`). The trigger day is set a few days *before* the median reorder, so the recommendation lands while the customer is still deciding — turning a static rule into a timed flow.

In [5]:
reorder = pd.read_csv(EDA_OUTPUTS / "12_reorder_interval_by_sku.csv")
reorder["handle"] = reorder["flavor_label"].astype(str).str.split(" / ").str[0].str.strip()
reorder = reorder[reorder["handle"].str.len() > 0]
handle_reorder = (reorder.groupby("handle")
                  .apply(lambda g: pd.Series({
                      "repeat_buyers": int(g["repeat_buyers"].sum()),
                      "median_reorder_days": np.average(g["median_reorder_days"],
                                                        weights=g["repeat_buyers"]),
                  }), include_groups=False)
                  .reset_index()
                  .sort_values("repeat_buyers", ascending=False))

# Best cross-sell per anchor handle, taken from the SEQUENTIAL (next-purchase)
# rules -- the right lens for replenishment, since we want "what do they reach
# for *after* this one". We exclude same-handle transitions (that is just the
# reorder itself) and pick the strongest graduation partner by next-order lift.
seq_h = seq_handle[seq_handle["antecedent"] != seq_handle["consequent"]].copy()
best_xsell = (seq_h.sort_values(["next_order_lift", "co_customers"],
                                ascending=[False, False])
              .groupby("antecedent").first().reset_index()
              [["antecedent", "consequent", "next_order_confidence", "next_order_lift"]]
              .rename(columns={"antecedent": "handle",
                               "consequent": "recommended_cross_sell"}))

triggers = handle_reorder.merge(best_xsell, on="handle", how="left")
triggers["trigger_day"] = (triggers["median_reorder_days"] - 7).round().clip(lower=7)
triggers["play"] = np.where(
    triggers["recommended_cross_sell"].notna(),
    "Day " + triggers["trigger_day"].astype("Int64").astype(str)
        + ": remind to reorder " + triggers["handle"]
        + " + introduce " + triggers["recommended_cross_sell"].fillna(""),
    "Day " + triggers["trigger_day"].astype("Int64").astype(str)
        + ": reorder reminder for " + triggers["handle"] + " (no strong cross-sell yet)")
triggers.to_csv(MBA_OUTPUTS / "replenishment_triggers.csv", index=False)
print("Replenishment-trigger calendar (top hero handles):")
display(triggers.head(12).reset_index(drop=True))

Replenishment-trigger calendar (top hero handles):


,handle,repeat_buyers,median_reorder_days,recommended_cross_sell,next_order_confidence,next_order_lift,trigger_day,play
0,collagen-glow,189.000,47.029,micronized-creatine-monohydrate,0.106,0.942,40.000,Day 40: remind to reorder collagen-glow + intr...
1,better-whey,178.000,70.927,micronized-creatine-monohydrate,0.091,0.811,64.000,Day 64: remind to reorder better-whey + introd...
2,clear-protein,160.000,52.962,discovery-sampler,0.034,2.834,46.000,Day 46: remind to reorder clear-protein + intr...
3,lean-protein,136.000,36.176,clear-protein-25g-single-sachet,0.060,1.989,29.000,Day 29: remind to reorder lean-protein + intro...
4,plant-protein,102.000,54.250,soy-protein-isolate,0.083,1.274,47.000,Day 47: remind to reorder plant-protein + intr...
5,micronized-creatine-monohydrate,82.000,63.250,collagen-glow,0.149,1.062,56.000,Day 56: remind to reorder micronized-creatine-...
6,prime-whey-isolate,72.000,50.389,better-whey,0.193,0.895,43.000,Day 43: remind to reorder prime-whey-isolate +...
7,soy-protein-isolate,50.000,67.600,plant-protein,0.105,1.178,61.000,Day 61: remind to reorder soy-protein-isolate ...
8,lushprotein-clear-shaker,35.000,50.286,clear-protein-25g-single-sachet,0.066,2.186,43.000,Day 43: remind to reorder lushprotein-clear-sh...
9,pureburn-fat-burner-capsules,11.000,74.545,NaN,NaN,NaN,68.000,Day 68: reorder reminder for pureburn-fat-burn...


## Readout: timing and sequence change the answer

**Sequential rules surface graduation paths within-basket MBA cannot see.** The pairs printed in section 1 that appear *only* in the next-purchase view are exactly the journeys to engineer with post-purchase email — they describe how customers naturally expand, not just what they grab in one go.

**The gateway scorecard tells acquisition what to sell first.** Entry products differ sharply in the loyalty they pre-load (repeat rate and subscription propensity vary several-fold across first categories/flavors). LushProtein should bias paid acquisition and first-order offers toward the high-gateway-score entry points, because the cheapest retention is choosing the right front door.

**The replenishment calendar converts rules into a schedule.** Because hero SKUs reorder on a measured ~6–8 week clock, each affinity rule becomes a dated trigger: remind-to-reorder + a retention-scored cross-sell, fired ~7 days before run-out. This is the operational bridge between MBA and a Klaviyo/Recharge flow.

Outputs written: `mba_sequential_rules.csv`, `gateway_product_scorecard.csv`, `replenishment_triggers.csv`. Notebook `05` turns all of this into named campaigns.

---
### Final metrics & scores

**Sequential (next-order) rules** — 2,223 repeat customers analysed; `lift` > 1 = a genuine graduation path:

| First → Next (handle) | Customers | Next-order conf. | Lift |
|---|--:|--:|--:|
| lean 40g single-serve → lean-protein (full) | 82 | 68% | 2.97 |
| clear-protein → discovery-sampler | 19 | 3% | 2.83 |
| clear-shaker → clear 25g sachet | 28 | 7% | 2.19 |
| lean-protein → clear 25g sachet | 25 | 6% | 1.99 |
| clear 25g sachet → lean-protein | 30 | 45% | 1.96 |

**Gateway flavors** (`gateway_score` = repeat × subscription, min 30 first-buyers):

| Entry flavor | First-buyers | Repeat | Subscribed | Gateway score |
|---|--:|--:|--:|--:|
| lean-protein — Taro 1kg | 39 | 38% | 21% | 0.46 |
| collagen-glow — 300g | 76 | 34% | 20% | 0.41 |
| lean-protein — Thai Milk Tea 1kg | 192 | 35% | 18% | 0.41 |

*(Tiny-n entries showing 100% repeat — e.g. Better Whey Cocoa Dinosaur, n=30 — are flagged as over-fit and kept out of planning.)*

**Replenishment-trigger calendar** (fire ~7d before median run-out; cross-sell shown with its next-order lift):

| Anchor handle | Reorder (median d) | Trigger day | Cross-sell (lift) |
|---|--:|--:|---|
| lean 40g single-serve | 12 | 7 | lean-protein (2.97) |
| lean-protein | 36 | 29 | clear 25g sachet (1.99) |
| collagen-glow | 47 | 40 | creatine (0.94) |
| prime-whey-isolate | 50 | 43 | better-whey (0.89) |
| clear-shaker | 50 | 43 | clear 25g sachet (2.19) |
| clear-protein | 53 | 46 | discovery-sampler (2.83) |

**Read:** single-serve trials graduate to full-size at **~3× lift**, and hero proteins reorder on a **~5–7 week clock** — the timing that converts a static affinity rule into a triggered flow.
